# Does the MONTH BODY (day ~3–22) momentum carry through the turn into next month's 1st?

Not prior-month — SAME month. Let the day-1 pop wear off, measure the **body** (day 3→22, excluding the
first 2 days and the last 2), and ask: **if the body is UP, does that momentum carry through** (a) the
**tail of the same month** (last 2 days), (b) the **turn** (last 2 of this month + first 2 of next), and
(c) **next month's day-1** specifically?

Panels: 1) body UP vs DOWN -> tail / next-day1 / turn / next-body.  2) correlations.  3) body QUINTILES ->
next-day1 (monotonicity check — the Q4 discipline).  4) verdict read.  Run top-to-bottom; last cell exports.


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np
def load_spx():
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    try:
        df = yf.download('^GSPC', start='1990-01-01', progress=False, auto_adjust=True)
        if len(df):
            s = df['Close']; s = s.iloc[:,0] if hasattr(s,'columns') else s
            return pd.Series(np.asarray(s).ravel(), index=df.index, name='spx').dropna()
    except Exception as e: print('yf failed -> Stooq:', e)
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('spx').dropna()
spx = load_spx()
d = pd.DataFrame({'px': spx}); d['ret'] = d['px'].pct_change(); d = d.dropna()
d['ym'] = d.index.to_period('M'); g = d.groupby('ym')
d['dom_start'] = g.cumcount() + 1
d['dom_end']   = g['px'].transform('size') - g.cumcount() - 1
print('SPX', len(d), 'days', d.index[0].date(), '->', d.index[-1].date())


## Build monthly features: body (3–22), tail (last 2), first-2, day-1


In [ ]:
def cum(s): return (1+s).prod()-1
day1  = d[d['dom_start']==1].groupby('ym')['ret'].first()
# BODY = day 3..22, but never the last 2 days (so it can't swallow the turn)
body  = d[(d['dom_start']>=3)&(d['dom_start']<=22)&(d['dom_end']>=2)].groupby('ym')['ret'].apply(cum)
tail2 = d[d['dom_end']<=1].groupby('ym')['ret'].apply(cum)      # last 2 days = 'rest of that month'
first2= d[d['dom_start'].isin([1,2])].groupby('ym')['ret'].apply(cum)
M = pd.DataFrame({'day1':day1,'body':body,'tail2':tail2,'first2':first2}).dropna(subset=['body'])
M = M.sort_index()
M['next_day1'] = M['day1'].shift(-1)
M['next_body'] = M['body'].shift(-1)
# TURN = last 2 of THIS month chained with first 2 of NEXT month
M['turn'] = (1+M['tail2'])*(1+M['first2'].shift(-1)) - 1
def s(r): r=r.dropna(); return f'n={len(r):4d}  mean {1e4*r.mean():+7.1f}bp  win {100*(r>0).mean():4.0f}%  med {1e4*r.median():+7.1f}bp'
print('months with a body:', M['body'].notna().sum())


## P1 — body UP vs DOWN: does momentum carry to tail / next-day1 / turn / next-body?


In [ ]:
up, dn = M['body']>0, M['body']<=0
print('BODY up  ->  same-month TAIL (last2):', s(M.loc[up,'tail2']))
print('BODY down->  same-month TAIL (last2):', s(M.loc[dn,'tail2']))
print('BODY up  ->  NEXT-month DAY-1       :', s(M.loc[up,'next_day1']))
print('BODY down->  NEXT-month DAY-1       :', s(M.loc[dn,'next_day1']))
print('BODY up  ->  TURN (last2+next first2):', s(M.loc[up,'turn']))
print('BODY down->  TURN (last2+next first2):', s(M.loc[dn,'turn']))
print('BODY up  ->  NEXT-month BODY        :', s(M.loc[up,'next_body']))
print('BODY down->  NEXT-month BODY        :', s(M.loc[dn,'next_body']))
print('\nMomentum CARRIES if UP rows > DOWN rows on the carry targets; REVERSES if UP < DOWN.')


## P2 — correlations (does body strength scale the carry?)


In [ ]:
for tgt in ['tail2','turn','next_day1','next_body']:
    c = M['body'].corr(M[tgt])
    print(f'  corr(body, {tgt:9s}) = {c:+.2f}')
print('\n~0 = body does not predict it; + = continuation; - = reversal. (|corr|>~0.1 on n~430 is the bar.)')


## P3 — body QUINTILES -> next-month day-1 (monotonic, or another Q4 noise-hole?)


In [ ]:
M2 = M.dropna(subset=['next_day1']).copy()
M2['q'] = pd.qcut(M2['body'], 5, labels=['Q1 worst','Q2','Q3','Q4','Q5 best'])
for q in ['Q1 worst','Q2','Q3','Q4','Q5 best']:
    print(f'  body {q:9s} -> next day1:', s(M2.loc[M2['q']==q,'next_day1']))
print('\nMonotone rising = real continuation; a broken middle = small-sample noise (like prior-month Q4).')


## P4 — same, but body -> TURN by quintile (the fuller carry window)


In [ ]:
M3 = M.dropna(subset=['turn']).copy()
M3['q'] = pd.qcut(M3['body'], 5, labels=['Q1 worst','Q2','Q3','Q4','Q5 best'])
for q in ['Q1 worst','Q2','Q3','Q4','Q5 best']:
    print(f'  body {q:9s} -> turn:', s(M3.loc[M3['q']==q,'turn']))
print('\nBaseline for reference:')
print('  ALL months  turn:', s(M['turn'])); print('  ALL next_day1:', s(M['next_day1']))


## ⬇️ EXPORT — run LAST


In [ ]:
from datetime import datetime
fname='body_momentum_report.txt'
hdr=['='*66,'MONTH-BODY MOMENTUM -> TURN/NEXT-1st — export',
     f'SPX {d.index[0].date()} -> {d.index[-1].date()} | {len(d)} days | {len(M)} months',
     f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*66,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')
